---
title: "DRG Grouping v2"

author: "Carlos Resurreccion"

date: "2024-10-21"
---


# Parameters

Change which year_to_load to process in
`~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt`

Change other rarely touched parameters in
`~/pids-drg-claims/data-cleaning/r_scripts_v2/00_v2_params-fpaths.R`


In [1]:
source(here::here("data-cleaning", "00a-parameters.r"))
year_range <- c(2018:2023)


Parallelization: TRUE 


# Libraries


In [2]:
source(here::here("data-cleaning", "00b-packages.r"))


Loading required package: data.table

Loading required package: here

here() starts at /home/resurreccion_cmc/pids-drg-claims

Loading required package: tictoc


Attaching package: ‘tictoc’


The following object is masked from ‘package:data.table’:

    shift


Loading required package: stringr

Loading required package: stringi

Loading required package: lubridate


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Loading required package: profvis

Loading required package: hash

hash-2.2.6.3 provided by Decision Patterns



Attaching package: ‘hash’


The following object is masked from ‘package:tictoc’:

    clear


The following object is masked from ‘package:data.table’:

    copy


Loading required package: future

Loading required package: future.apply

Load

# R Scripts


In [3]:
source(here::here("data-cleaning", "00c-load-params-and-scripts.r"))


==== Loaded Parameters ====

year_to_load: 2023

Automate: FALSE


Sourcing scripts from:/home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R

✅ Authentication successful using service account key.

All directories exist.


Total Rows via cached object: 12996898

Utilizing 16 cores (32 threads)


Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.2.0.process_helper_functions.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/0.3.0.grouping_functions.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/1.0.query_bq_to_dt.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/2.0.split_and_save_part.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts_v2/3.0.create_sample_files.R

Sourcing: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/r_scripts

# Load Mapping Data


In [4]:
source(here::here("data-cleaning", "00d-load-mapping.r"))
# hci_query <- paste0("SELECT * FROM ", gcp_proj, ".phic_hfac.hfac_", year_to_load)
# hci <- load_or_query(hci_query, "hci", year_to_load)


==== Loading Data Mapping ====

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/proc.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/rvs_icd9.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/acr_rvs.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/tdrg_icd10.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/phl_icd10.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/i10vx.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/zben.rds

Loading from cache: /home/resurreccion_cmc/pids-drg-claims/data-cleaning/debug/cache/mapping/2025/acr.rds

00d-load-mapping.r successfully executed.



# Subset Generation


In [5]:
# Prompt for manual confirmation if needed
to_gen_subset <- tolower(readline(
  prompt = "Generate Subset with Ageday and Bwt for Thai and Python? (y/n): "
))
if (to_gen_subset != "y") {
  to_gen_subset <- FALSE
  message("Skipping Subset Generation\n")
} else {
  to_gen_subset <- TRUE
}
# Prompt for manual confirmation if needed
to_thai_all_years <- tolower(readline(
  prompt = "Do all thai years? (y/n): "
))
if (to_thai_all_years != "y") {
  message("Processing single year only\n")
  to_thai_all_years <- FALSE
} else {
  to_thai_all_years <- TRUE
}


In [6]:
if (to_gen_subset) {
  if (!to_thai_all_years) {
    cat("\rReading final\n")
    flush.console()
    result <- readRDS(here(
      chkpt_2_path,
      paste0(chkpt_2_prefix, year_to_load, suffix, "v2", "_part_a_is_covid", ".rds")
    ))

    print("Total Rows")
    print(nrow(result))
    print("Invalid PDx")
    print(nrow(result[clin_pdx_source == 99]))
    print("Birthdates to be imputed")
    print(nrow(result[is.na(pat_bdate)]))
    print("NA age")
    print(nrow(result[is.na(pat_age)]))
    result <- result[clin_outpatient == FALSE]
    print("Inpatient Rows")
    print(nrow(result))
    result <- result[is_covid == FALSE]
    print("Inpatient Non-Covid Rows")
    print(nrow(result))
    print("Inpatient Non-Covid Rows from INF, L1, L2, L3 only")
    result <- result[id_hci %chin% hci[inst_level %chin% c("INF", "L1", "L2", "L3"), id_hci]]
    print(nrow(result))

    cat("\rComputing pat_bdate\n")
    # Impute missing birthdates (pat_bdate) based on
    # admission date (date_adm) and age (pat_age)
    result[
      (!is.na(pat_age) & is.na(pat_bdate) & !is.na(date_adm)) &
        as.Date(date_adm) - round(pat_age * 365.25) >= as.Date("1900-01-01"),
      pat_bdate := as.Date(date_adm) - round(pat_age * 365.25)
    ]

    cat("\rComputing pat_ageday\n")
    result[, pat_ageday := NA_real_]
    result[
      !is.na(pat_age) & pat_age >= 0 & pat_age < 1 &
        !is.na(date_adm) & !is.na(pat_bdate) & is.na(pat_ageday),
      pat_ageday := as.integer(difftime(as.Date(format(date_adm, "%Y-%m-%d")),
        as.Date(pat_bdate),
        units = "days"
      ))
    ]
    result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 &
      (pat_ageday > 365 | pat_ageday < 0), pat_ageday := 0]
    result[
      !is.na(pat_age) & pat_age >= 0 & pat_age < 1 & (pat_ageday == 365),
      `:=`(
        pat_ageday = 364, # Update pat_ageday to 364
        pat_bdate = pat_bdate + 1 # Add 1 day to pat_bdate
      )
    ]
    cat("\rFlooring pat_ageday\n")
    result[!is.na(pat_ageday), pat_ageday := as.integer(floor(pat_ageday))]

    cat("\rComputing pat_bwt\n")
    set.seed(global_seed)
    bw_dist <- c(
      # Random bwt between 0.5 and 0.9 for 2 newborns
      round(runif(2, 0.5, 0.9), 3),
      # Random bwt between 1.1 and 1.4 for 8 newborns
      round(runif(8, 1.1, 1.4), 3),
      # Random bwt between 1.6 and 1.9 for 19 newborns
      round(runif(19, 1.6, 1.9), 3),
      # Random bwt between 2.1 and 2.4 for 95 newborns
      round(runif(95, 2.1, 2.4), 3),
      # Random bwt between 2.6 and 2.9 for 381 newborns
      round(runif(381, 2.6, 2.9), 3),
      # Random bwt between 3.1 and 3.4 for 375 newborns
      round(runif(375, 3.1, 3.4), 3),
      # Random bwt between 3.5 and 4.0 for 115 newborns
      round(runif(115, 3.5, 4.0), 3),
      # Random bwt between 0.5 and 4.0 for 6 newborns
      round(runif(6, 0.5, 4.0), 3)
    )

    # Create the zero_mask condition where pat_age is between 0 and 1 (newborns)
    zero_mask <- result[, pat_age >= 0 & pat_age < 1]

    # Apply bwt only if pat_bwt is NA and zero_mask is TRUE
    result[(is.na(pat_bwt) | pat_bwt <= 0) & zero_mask,
      pat_bwt := sapply(.SD$pat_bwt, \(x) sample(bw_dist, 1)),
      .SDcols = "pat_bwt"
    ]
    result[!zero_mask, pat_bwt := NA_real_]

    cat("\rWriting final\n")
    flush.console()
    saveRDS(result, here(
      chkpt_2_path,
      paste0(chkpt_2_prefix, year_to_load, suffix, "v2_part_c_ageday_bwt", ".rds")
    ))
    # Check for duplicates in id_series
    if (any(duplicated(result$id_series))) {
      # Identify duplicates
      duplicate_ids <- result$id_series[duplicated(result$id_series)]

      # Extract rows with duplicate id_series
      duplicate_rows <- result[id_series %in% duplicate_ids, ]

      # Print rows with duplicates
      cat("Rows with duplicate 'id_series':\n")
      print(duplicate_rows)

      # Stop execution
      stop("The 'id_series' column contains duplicates. Execution stopped.")
    }
    print("Deduplicated id_series")
    print(nrow(result))

    # Check for duplicates in id_series
    if (any(duplicated(result$caseid))) {
      # Identify duplicates
      duplicate_ids <- result$caseid[duplicated(result$caseid)]

      # Extract rows with duplicate id_series
      duplicate_rows <- result[caseid %in% duplicate_ids, ]

      # Print rows with duplicates
      cat("Rows with duplicate 'caseid':\n")
      print(duplicate_rows)

      # Stop execution
      stop("The 'caseid' column contains duplicates. Execution stopped.")
    }
    print("Deduplicated caseid")
    print(nrow(result))
    print("Invalid PDx")
    print(nrow(result[clin_pdx_source == 99]))
    print("NA birthdate")
    print(nrow(result[is.na(pat_bdate)]))
    print("NA age")
    print(nrow(result[is.na(pat_age)]))
    if (any(!is.na(unique(result$clin_pdx[result$clin_pdx_source == 99])))) {
      stop("An invalid clin_pdx values is not NA, which is unexpected.")
    }
  } else if (to_thai_all_years) {
    # for (year_to_load in year_range) {
    for (year_to_load in year_range) {
      cat("\rReading final\n")
      flush.console()
      result <- readRDS(here(
        chkpt_2_path,
        paste0(chkpt_2_prefix, year_to_load, suffix, "v2", "_part_a_is_covid", ".rds")
      ))

      hci_query <- paste0("SELECT * FROM ", gcp_proj, ".phic_hfac.hfac_", year_to_load)
      hci <- load_or_query(hci_query, "hci", year_to_load)

      print("Total Rows")
      print(nrow(result))
      print("Invalid PDx")
      print(nrow(result[clin_pdx_source == 99]))
      print("Birthdates to be imputed")
      print(nrow(result[is.na(pat_bdate)]))
      print("NA ages")
      print(nrow(result[is.na(pat_age)]))
      result <- result[clin_outpatient == FALSE]
      print("Inpatient Rows")
      print(nrow(result))
      result <- result[is_covid == FALSE]
      print("Inpatient Non-Covid Rows")
      print(nrow(result))
      print("Inpatient Non-Covid Rows from INF, L1, L2, L3 only")
      result <- result[id_hci %chin% hci[inst_level %chin% c("INF", "L1", "L2", "L3"), id_hci]]
      print(nrow(result))

      cat("\rComputing pat_bdate\n")
      # Impute missing birthdates (pat_bdate) based on
      # admission date (date_adm) and age (pat_age)
      result[
        (!is.na(pat_age) & is.na(pat_bdate) & !is.na(date_adm)) &
          as.Date(date_adm) - round(pat_age * 365.25) >= as.Date("1900-01-01"),
        pat_bdate := as.Date(date_adm) - round(pat_age * 365.25)
      ]

      cat("\rComputing pat_ageday\n")
      result[, pat_ageday := NA_real_]
      result[
        !is.na(pat_age) & pat_age >= 0 & pat_age < 1 &
          !is.na(date_adm) & !is.na(pat_bdate) & is.na(pat_ageday),
        pat_ageday := as.integer(difftime(as.Date(format(date_adm, "%Y-%m-%d")),
          as.Date(pat_bdate),
          units = "days"
        ))
      ]
      result[!is.na(pat_age) & pat_age >= 0 & pat_age < 1 &
        (pat_ageday > 365 | pat_ageday < 0), pat_ageday := 0]
      result[
        !is.na(pat_age) & pat_age >= 0 & pat_age < 1 & (pat_ageday == 365),
        `:=`(
          pat_ageday = 364, # Update pat_ageday to 364
          pat_bdate = pat_bdate + 1 # Add 1 day to pat_bdate
        )
      ]
      cat("\rFlooring pat_ageday\n")
      result[!is.na(pat_ageday), pat_ageday := as.integer(floor(pat_ageday))]

      cat("\rComputing pat_bwt\n")
      set.seed(global_seed)
      bw_dist <- c(
        # Random bwt between 0.5 and 0.9 for 2 newborns
        round(runif(2, 0.5, 0.9), 3),
        # Random bwt between 1.1 and 1.4 for 8 newborns
        round(runif(8, 1.1, 1.4), 3),
        # Random bwt between 1.6 and 1.9 for 19 newborns
        round(runif(19, 1.6, 1.9), 3),
        # Random bwt between 2.1 and 2.4 for 95 newborns
        round(runif(95, 2.1, 2.4), 3),
        # Random bwt between 2.6 and 2.9 for 381 newborns
        round(runif(381, 2.6, 2.9), 3),
        # Random bwt between 3.1 and 3.4 for 375 newborns
        round(runif(375, 3.1, 3.4), 3),
        # Random bwt between 3.5 and 4.0 for 115 newborns
        round(runif(115, 3.5, 4.0), 3),
        # Random bwt between 0.5 and 4.0 for 6 newborns
        round(runif(6, 0.5, 4.0), 3)
      )

      # Create the zero_mask condition where
      # pat_age is between 0 and 1 (newborns)
      zero_mask <- result[, pat_age >= 0 & pat_age < 1]

      # Apply bwt only if pat_bwt is NA and zero_mask is TRUE
      result[(is.na(pat_bwt) | pat_bwt <= 0) & zero_mask,
        pat_bwt := sapply(.SD$pat_bwt, \(x) sample(bw_dist, 1)),
        .SDcols = "pat_bwt"
      ]
      result[!zero_mask, pat_bwt := NA_real_]

      cat("\rWriting final\n")
      flush.console()
      saveRDS(result, here(
        chkpt_2_path,
        paste0(chkpt_2_prefix, year_to_load, suffix, "v2_part_c_ageday_bwt", ".rds")
      ))
      # Check for duplicates in id_series
      if (any(duplicated(result$id_series))) {
        # Identify duplicates
        duplicate_ids <- result$id_series[duplicated(result$id_series)]

        # Extract rows with duplicate id_series
        duplicate_rows <- result[id_series %in% duplicate_ids, ]

        # Print rows with duplicates
        cat("Rows with duplicate 'id_series':\n")
        print(duplicate_rows)

        # Stop execution
        stop("The 'id_series' column contains duplicates. Execution stopped.")
      }
      print("Deduplicated id_series")
      print(nrow(result))

      # Check for duplicates in id_series
      if (any(duplicated(result$caseid))) {
        # Identify duplicates
        duplicate_ids <- result$caseid[duplicated(result$caseid)]

        # Extract rows with duplicate id_series
        duplicate_rows <- result[caseid %in% duplicate_ids, ]

        # Print rows with duplicates
        cat("Rows with duplicate 'caseid':\n")
        print(duplicate_rows)

        # Stop execution
        stop("The 'caseid' column contains duplicates. Execution stopped.")
      }
      print("Deduplicated caseid")
      print(nrow(result))
      print("Invalid PDx")
      print(nrow(result[clin_pdx_source == 99]))
      print("NA birthdate")
      print(nrow(result[is.na(pat_bdate)]))
      print("NA age")
      print(nrow(result[is.na(pat_age)]))
      if (any(!is.na(unique(result$clin_pdx[result$clin_pdx_source == 99])))) {
        stop("An invalid clin_pdx values is not NA, which is unexpected.")
      }
    }
  }
}


Reading final


Querying: hci



[1] "Total Rows"
[1] 10624981
[1] "Invalid PDx"
[1] 1291408
[1] "Birthdates to be imputed"
[1] 10624981
[1] "NA ages"
[1] 70747
[1] "Inpatient Rows"
[1] 7534712
[1] "Inpatient Non-Covid Rows"
[1] 7534710
[1] "Inpatient Non-Covid Rows from INF, L1, L2, L3 only"
[1] 6983185
Computing pat_bdate
Computing pat_ageday
Flooring pat_ageday
Computing pat_bwt
Writing final
[1] "Deduplicated id_series"
[1] 6983185
[1] "Deduplicated caseid"
[1] 6983185
[1] "Invalid PDx"
[1] 777619
[1] "NA birthdate"
[1] 64016
[1] "NA age"
[1] 64012
Reading final


Querying: hci



[1] "Total Rows"
[1] 12428391
[1] "Invalid PDx"
[1] 972813
[1] "Birthdates to be imputed"
[1] 12428391
[1] "NA ages"
[1] 89573
[1] "Inpatient Rows"
[1] 8656675
[1] "Inpatient Non-Covid Rows"
[1] 8656670
[1] "Inpatient Non-Covid Rows from INF, L1, L2, L3 only"
[1] 8030221
Computing pat_bdate
Computing pat_ageday
Flooring pat_ageday
Computing pat_bwt
Writing final
[1] "Deduplicated id_series"
[1] 8030221
[1] "Deduplicated caseid"
[1] 8030221
[1] "Invalid PDx"
[1] 660777
[1] "NA birthdate"
[1] 77463
[1] "NA age"
[1] 77461
Reading final


...

[1] "Total Rows"
[1] 13209411
[1] "Invalid PDx"
[1] 1824674
[1] "Birthdates to be imputed"
[1] 13209411
[1] "NA ages"
[1] 67447
[1] "Inpatient Rows"
[1] 4772978
[1] "Inpatient Non-Covid Rows"
[1] 4423353
[1] "Inpatient Non-Covid Rows from INF, L1, L2, L3 only"
[1] 3864561
Computing pat_bdate
Computing pat_ageday
Flooring pat_ageday
Computing pat_bwt
Writing final
[1] "Deduplicated id_series"
[1] 3864561
[1] "Deduplicated caseid"
[1] 3864561
[1] "Invalid PDx"
[1] 188563
[1] "NA birthdate"
[1] 29968
[1] "NA age"
[1] 29968
Reading final


Querying: hci



[1] "Total Rows"
[1] 12569941
[1] "Invalid PDx"
[1] 1153034
[1] "Birthdates to be imputed"
[1] 12569941
[1] "NA ages"
[1] 51338
[1] "Inpatient Rows"
[1] 5759244
[1] "Inpatient Non-Covid Rows"
[1] 5680492
[1] "Inpatient Non-Covid Rows from INF, L1, L2, L3 only"
[1] 5147682
Computing pat_bdate
Computing pat_ageday
Flooring pat_ageday
Computing pat_bwt
Writing final
[1] "Deduplicated id_series"
[1] 5147682
[1] "Deduplicated caseid"
[1] 5147682
[1] "Invalid PDx"
[1] 271985
[1] "NA birthdate"
[1] 33846
[1] "NA age"
[1] 33844
Reading final


Querying: hci



[1] "Total Rows"
[1] 12996898
[1] "Invalid PDx"
[1] 583936
[1] "Birthdates to be imputed"
[1] 12996898
[1] "NA ages"
[1] 54039
[1] "Inpatient Rows"
[1] 6758802
[1] "Inpatient Non-Covid Rows"
[1] 6736466
[1] "Inpatient Non-Covid Rows from INF, L1, L2, L3 only"
[1] 6230833
Computing pat_bdate
Computing pat_ageday
Flooring pat_ageday
Computing pat_bwt
Writing final
[1] "Deduplicated id_series"
[1] 6230833
[1] "Deduplicated caseid"
[1] 6230833
[1] "Invalid PDx"
[1] 220469
[1] "NA birthdate"
[1] 43173
[1] "NA age"
[1] 43173


# Python


In [7]:
# if (to_python && !to_generate_subset && to_generate_feather) {
#   cat("\rReading final\n")
#   flush.console()
#   result <- readRDS(here(chkpt_2_path, paste0(chkpt_2_prefix, year_to_load, suffix, "v2_part_c_ageday_bwt", ".rds")))
# }

# if (to_python && to_generate_feather) {
#   message("Renaming columns")
#   # Convert relevant data types
#   result[, patage := as.numeric(pat_age)]
#   result[, patsex := as.character(pat_sex)]
#   result[, birthweight := as.numeric(pat_bwt)]
#   result[, discharge := as.integer(clin_discharge)]
#   result[, dob := as.Date(pat_bdate)]
#   result[, ageday := as.integer(pat_ageday)]
#   result[, pdx := clin_pdx]

#   # Handle ICD splitting for columns that are lists of character vectors
#   split_codes_from_list <- function(dt, column, prefix, max_cols) {
#     split_list <- dt[[column]] # Extract the list column
#     # Pad each list to the specified max_cols with NAs if not enough elements
#     # split_cols <- lapply(1:max_cols, \(i) sapply(split_list, \(x) if (length(x) >= i) x[[i]] else NA_character_))
#     # Use mclapply for parallel processing
#     split_cols <- parallel::mclapply(
#       1:max_cols,
#       \(i) sapply(split_list, \(x) if (length(x) >= i) x[[i]] else NA_character_),
#       mc.cores = nthreads # Automatically use all available cores
#     )

#     split_dt <- as.data.table(split_cols)
#     setnames(split_dt, paste0(prefix, 1:max_cols))
#     return(split_dt)
#   }

#   # Apply the function to split clin_sdx and clin_proc
#   message("Splitting clin_sdx")
#   sdx_columns <- split_codes_from_list(result, "clin_sdx", "sdx", 12)
#   message("Splitting clin_proc")
#   proc_columns <- split_codes_from_list(result, "clin_proc", "proc", 20)

#   # Combine the split columns back into the result
#   message("cbind results")
#   result <- cbind(result, sdx_columns, proc_columns)

#   # Replace NA in non-date columns with "None"
#   # non_date_columns <- c("patsex", "pdx", paste0("sdx", 1:12), paste0("proc", 1:20))
#   # result[, (non_date_columns) := lapply(.SD, \(x) ifelse(is.na(x), "None", x)), .SDcols = non_date_columns]

#   # Prepare the final data table for writing
#   for_fwrite <- result[, c(
#     "id_series", "date_adm", "date_dis", "time_adm", "time_dis", "patage", "dob", "patsex", "discharge", "pdx",
#     paste0("sdx", 1:12), paste0("proc", 1:20), "birthweight", "ageday"
#   ), with = FALSE]
#   message("Formatting date_adm")
#   for_fwrite[, date_adm := format(date_adm, "%Y-%m-%d %H:%M:%S")]
#   message("Formatting date_dis")
#   for_fwrite[, date_dis := format(date_dis, "%Y-%m-%d %H:%M:%S")]

#   for_fwrite[, time_adm := NULL]
#   for_fwrite[, time_dis := NULL]

#   # Write the final table to a CSV file
#   message("Writing to csv")
#   fwrite(for_fwrite, here(chkpt_7_path, paste0(chkpt_7b_prefix, suffix, ".csv")))
#   # message("Creating summary table")
#   # # Create a summary table that shows the count of non-null values for each column
#   # summary_table <- for_fwrite[, lapply(.SD, \(x) sum(!is.na(x))), .SDcols = names(for_fwrite)]

#   # # Transpose the summary table to make it more readable
#   # summary_table <- transpose(summary_table)
#   # setnames(summary_table, "Non-Null Count")
#   # summary_table[, Column := names(for_fwrite)]

#   # # Reorder the summary table to show the columns
#   # setcolorder(summary_table, c("Column", "Non-Null Count"))

#   # # Print the summary table
#   # message("Printing summary table")
#   # print(summary_table)
#   saveRDS(for_fwrite, here(chkpt_7_path, paste0("for_fwrite_", year_to_load, suffix, ".rds")))
# }


In [8]:
# if (to_python) {
#   if (!to_generate_py_fwrite && to_generate_feather) for_fwrite <- readRDS(here(chkpt_7_path, paste0("for_fwrite_", year_to_load, suffix, ".rds")))
#   if (to_generate_feather) write_feather(as.data.frame(for_fwrite), here(chkpt_7_path, paste0("python_input_", year_to_load, suffix, ".feather")))

#   # Prompt for manual confirmation if needed
#   if (to_py_prompt) {
#     response <- tolower(readline(prompt = "Have you run the Python grouper manually? (y/n): "))
#     if (response != "y") {
#       stop("Python Grouper not run yet. Script terminated. Continue on manually if necessary")
#     }
#     message("Continuing with the script...\n")
#   } else {
#     message("Python Grouper is assumed to have been run already. Continuing with the script...\n")
#   }

#   output_dt <- as.data.table(read_feather(here(chkpt_8_path, paste0("python_output_", year_to_load, suffix, ".feather"))))
# }


In [9]:
# if (to_python) {
#   # Rename columns to match required names if necessary
#   setnames(output_dt,
#     old = c("drg", "pdc", "pccl", "error_code", "warning_code"),
#     new = c("py_drg", "py_pdc", "py_pccl", "py_err", "py_warn"), skip_absent = TRUE
#   )

#   # Select only the required columns
#   required_columns <- c("id_series", "py_drg", "py_pdc", "py_pccl", "py_err", "py_warn")
#   output_dt <- output_dt[, ..required_columns]

#   # Adjust data types
#   output_dt[, py_drg := as.character(py_drg)]
#   output_dt[, py_pdc := as.character(py_pdc)]
#   output_dt[, py_pccl := as.numeric(py_pccl)]

#   # Convert 'py_err' and 'py_warn' to arrays (list of character vectors)
#   array_columns <- c("py_err", "py_warn")

#   process_error_warning_column <- function(col) {
#     lapply(col, \(x) {
#       # Flatten x to a character vector
#       x <- unlist(x)
#       x <- as.character(x)

#       # If x is NULL or length zero after unlisting, return character(0)
#       if (is.null(x) || length(x) == 0) {
#         return(character(0))
#       }

#       # Remove any NA values from x
#       x <- x[!is.na(x)]

#       # Remove any "None", "NA", or empty strings from x
#       x <- x[!(x %in% c("None", "NA", "NaN", ""))]

#       # If x is now length zero after cleaning, return character(0)
#       if (length(x) == 0) {
#         return(character(0))
#       }

#       # Now split each element of x by comma and optional whitespace
#       split_x <- unlist(strsplit(x, ",\\s*"))

#       # Remove any empty strings, "NA", or "None" from split_x
#       split_x <- split_x[!(split_x %in% c("", "NaN", "NA", "None")) & !is.na(split_x)]

#       # Return character(0) if split_x is empty after cleaning
#       if (length(split_x) == 0) {
#         return(character(0))
#       } else {
#         return(split_x)
#       }
#     })
#   }

#   # Apply the processing function to the columns
#   output_dt[, (array_columns) := mclapply(.SD, process_error_warning_column, mc.cores = nthreads), .SDcols = array_columns]

#   # Now 'output_dt' is your final result
#   # You can proceed to use 'output_dt' as needed

#   # Replace <NA> values in 'py_drg' and 'py_pdc' with character(0)
#   output_dt[, py_drg := ifelse(is.na(py_drg), "", py_drg)]
#   output_dt[, py_pdc := ifelse(is.na(py_pdc), "", py_pdc)]
#   # output_dt[, py_pccl := ifelse(is.nan(py_pccl), NA_real_, py_pccl)]

#   # For example, print the first few rows
#   # print(head(output_dt[id_series == 24465430]))
#   print(head(output_dt))
# }

# # Ensure output_dt is a data.table
# setDT(output_dt)

# # Filtering for py_drg codes that start with '26'
# filtered_dt <- output_dt[startsWith(py_drg, "26")]

# cat("\nPercent Ungroupable:\n")
# cat(round(nrow(filtered_dt) / nrow(output_dt) * 100, 1))
# cat(" %\n\n")

# # Filtering out empty lists in py_warn
# filtered_dt <- filtered_dt[lengths(py_warn) > 0]

# # Combine warning codes as strings for rows with multiple warnings
# warning_counts <- filtered_dt[, .(warning_code = sapply(
#   py_warn,
#   function(x) paste(sort(unique(x)), collapse = ", ")
# )), by = id_series][
#   , .N,
#   by = warning_code
# ][order(-N)]

# # Expanding py_err normally (no need to combine multiple error codes)
# error_counts <- filtered_dt[, .(error_code = unlist(py_err)), by = id_series][
#   , .N,
#   by = error_code
# ][order(-N)]

# # Print results
# cat("Most Common Warning Codes for py_drg 26___\n")
# print(warning_counts)

# cat("\nMost Common Error Codes for py_drg 26___\n")
# print(error_counts)


In [10]:
# if (to_python) {
#   fwrite(output_dt, here(chkpt_8_path, paste0("python_output_", year_to_load, suffix, ".csv")))
#   # str(output_dt)
#   saveRDS(output_dt, here(chkpt_8_path, paste0("python_output_", year_to_load, suffix, ".rds")))
# }


# Thai


## Input Prep


In [11]:
# Prompt for manual confirmation if needed
to_generate_thai_txt <- tolower(readline(
  prompt = "Generate Thai TXT Inputs? (y/n): "
))
if (to_generate_thai_txt != "y") {
  message("Skipping TXT Generation\n")
  to_generate_thai_txt <- FALSE
} else {
  to_generate_thai_txt <- TRUE
}


In [12]:
if (!to_thai_all_years) {
  if (to_generate_thai_txt) {
    cat("\rReading final\n")
    flush.console()
    result <- readRDS(here(
      chkpt_2_path,
      paste0(
        chkpt_2_prefix, year_to_load, suffix,
        "v2_part_c_ageday_bwt", ".rds"
      )
    ))
    # str(result)
    result[, caseid := as.character(seq_len(nrow(result)))]
    result_mapping <- result[, .(id_series, caseid)]
    cat("\rExporting for grouper\n")
    flush.console()
    # Define chunk size
    chunk_size <- 5000000
    num_chunks <- ceiling(nrow(result) / chunk_size)

    for (i in seq_len(num_chunks)) {
      # Define the file path and name for this part
      output_file <- here(
        chkpt_4_path,
        paste0(
          chkpt_4_prefix, year_to_load, suffix,
          "part_", i, "_of_", num_chunks, ".txt"
        )
      )

      # Extract the chunk
      start_row <- (i - 1) * chunk_size + 1
      end_row <- min(i * chunk_size, nrow(result))
      chunk <- result[start_row:end_row, ]

      # Export the chunk to a file
      export_for_grouper(chunk, output_file, i)
      message("Saved part ", i, " of ", num_chunks, " to ", output_file)

      # Upload the file to GCS
      message("Uploading part ", i, " of ", num_chunks, " to GCS")
      gcs_upload(
        file = output_file,
        bucket = gcs_bucket,
        name = paste0(gcs_pre_fpath, "/", basename(output_file)),
        predefinedAcl = "bucketLevel"
      )

      # Clean up memory
      rm(chunk)
      gc()
    }
  } else {
    message("Skipping thai txt generation")
  }
} else if (to_thai_all_years) {
  if (to_generate_thai_txt) {
    # for (year_to_load in year_range) {
    for (year_to_load in year_range) {
      cat("\rReading final\n")
      flush.console()
      result <- readRDS(here(
        chkpt_2_path,
        paste0(
          chkpt_2_prefix, year_to_load, suffix,
          "v2_part_c_ageday_bwt", ".rds"
        )
      ))
      # str(result)
      result[, caseid := as.character(seq_len(nrow(result)))]
      result_mapping <- result[, .(id_series, caseid)]
      cat("\rExporting for grouper\n")
      flush.console()
      # Define chunk size
      chunk_size <- 5000000
      num_chunks <- ceiling(nrow(result) / chunk_size)

      for (i in seq_len(num_chunks)) {
        # Define the file path and name for this part
        output_file <- here(
          chkpt_4_path,
          paste0(
            chkpt_4_prefix, year_to_load, suffix,
            "part_", i, "_of_", num_chunks, ".txt"
          )
        )

        # Extract the chunk
        start_row <- (i - 1) * chunk_size + 1
        end_row <- min(i * chunk_size, nrow(result))
        chunk <- result[start_row:end_row, ]

        # Export the chunk to a file
        export_for_grouper(chunk, output_file, i)
        message("Saved part ", i, " of ", num_chunks, " to ", output_file)

        # Upload the file to GCS
        message("Uploading part ", i, " of ", num_chunks, " to GCS")
        gcs_upload(
          file = output_file,
          bucket = gcs_bucket,
          name = paste0(gcs_pre_fpath, "/", basename(output_file)),
          predefinedAcl = "bucketLevel"
        )

        # Clean up memory
        rm(chunk)
        gc()
      }
    }
  } else {
    message("Skipping thai txt generation")
  }
}


Reading final
Exporting for grouper
Classes ‘data.table’ and 'data.frame':	5000000 obs. of  42 variables:
 $ CASEID : chr  "1" "2" "3" "4" ...
 $ DOB    : chr  "04/01/1999" "03/01/1947" "26/01/2002" "30/01/1985" ...
 $ Sex    : num  2 2 1 2 1 2 2 2 2 2 ...
 $ DateAdm: chr  "04/01/2018" "03/01/2018" "26/01/2018" "30/01/2018" ...
 $ TimeAdm: chr  "0400" "1945" "1526" "0830" ...
 $ DateDsc: chr  "06/01/2018" "04/01/2018" "29/01/2018" "01/02/2018" ...
 $ TimeDsc: chr  "1530" "2035" "1323" "0000" ...
 $ DischT : Factor w/ 5 levels "1","2","3","4",..: 1 1 1 1 2 1 1 1 1 1 ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "O996" "I119" "J683" "Z302" ...
 $ SDx1   : chr  "E86" "--" "--" "--" ...
 $ SDx2   : chr  "--" "--" "--" "--" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SD

Saved part 1 of 2 to /home/resurreccion_cmc/pids-drg-claims/data-cleaning/data/chkpts/2025/chkpt_4_thai_master_input/chkpt_4_thai_grouper_input_2018_full_part_1_of_2.txt

Uploading part 1 of 2 to GCS

ℹ 2025-05-06 04:22:22.872735 > File size detected as  769.1 Mb

ℹ 2025-05-06 04:22:22.979258 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/pids-drg-vm/o/?uploadType=resumable&name=data%2Fphic%2Fthai%2Fpre%2Fchkpt_4_thai_grouper_input_2018_full_part_1_of_2.txt&upload_id=AAO2VwrGiPe1rtGlWAVCbcNx717nzNFgflqr9IatwQHuRyz4DzRXkCIyub_JP2wkjPiu6Tnl2qXLcrQ1kOPqRDsgQvE_NXhGy7AGBSTtYyKx1Bs



Classes ‘data.table’ and 'data.frame':	1983185 obs. of  42 variables:
 $ CASEID : chr  "5000001" "5000002" "5000003" "5000004" ...
 $ DOB    : chr  "27/12/2008" "21/09/2012" "26/09/2014" "15/06/1993" ...
 $ Sex    : num  2 1 1 1 1 2 2 1 2 2 ...
 $ DateAdm: chr  "27/12/2018" "22/09/2018" "26/09/2018" "15/06/2018" ...
 $ TimeAdm: chr  "1520" "1152" "1517" "1105" ...
 $ DateDsc: chr  "04/01/2019" "25/09/2018" "29/09/2018" "18/06/2018" ...
 $ TimeDsc: chr  "1832" "1600" "1543" "1500" ...
 $ DischT : Factor w/ 5 levels "1","2","3","4",..: 1 1 1 1 1 1 1 1 1 2 ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "E878" "J069" "J459" "--" ...
 $ SDx1   : chr  "E835" "--" "--" "--" ...
 $ SDx2   : chr  "E871" "--" "--" "--" ...
 $ SDx3   : chr  "E880" "--" "--" "--" ...
 $ SDx4   : chr  "J90" "--" "--" "--" ...
 $ SDx5   : chr  "I313" "--" "--" "--" ...
 $ SDx6   : chr  "I411" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SDx9  

Saved part 2 of 2 to /home/resurreccion_cmc/pids-drg-claims/data-cleaning/data/chkpts/2025/chkpt_4_thai_master_input/chkpt_4_thai_grouper_input_2018_full_part_2_of_2.txt

Uploading part 2 of 2 to GCS

ℹ 2025-05-06 04:23:11.123116 > File size detected as  305.5 Mb

ℹ 2025-05-06 04:23:11.185524 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/pids-drg-vm/o/?uploadType=resumable&name=data%2Fphic%2Fthai%2Fpre%2Fchkpt_4_thai_grouper_input_2018_full_part_2_of_2.txt&upload_id=AAO2VwqtjWGkBVlVqGdKDvLJ3L_VoBDrYvZBAsgVkJ26vKay-F855jG7K-LljLdA9bm-zx_5L8rh2I2Wm6N6fyAg7Yp1EcmKNxCghykNwFexmg



Reading final
Exporting for grouper
Classes ‘data.table’ and 'data.frame':	5000000 obs. of  42 variables:
 $ CASEID : chr  "1" "2" "3" "4" ...
 $ DOB    : chr  "24/01/1965" "05/01/1953" "03/01/1993" "14/01/2019" ...
 $ Sex    : num  2 1 2 2 2 2 1 2 1 2 ...
 $ DateAdm: chr  "25/01/2019" "05/01/2019" "03/01/2019" "14/01/2019" ...
 $ TimeAdm: chr  "2230" "0300" "1453" "0515" ...
 $ DateDsc: chr  "27/01/2019" "08/01/2019" "06/01/2019" "15/01/2019" ...
 $ TimeDsc: chr  "1100" "1723" "1005" "1123" ...
 $ DischT : Factor w/ 5 levels "1","2","3","4",..: 1 1 1 1 1 1 2 1 1 1 ...
 $ AdmWt  : chr  "--" "--" "--" "1.368" ...
 $ PDx    : chr  "A099" "A158" "H46" "P831" ...
 $ SDx1   : chr  "E86" "--" "H332" "Z380" ...
 $ SDx2   : chr  "--" "--" "--" "--" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...

...

Saved part 2 of 2 to /home/resurreccion_cmc/pids-drg-claims/data-cleaning/data/chkpts/2025/chkpt_4_thai_master_input/chkpt_4_thai_grouper_input_2022_full_part_2_of_2.txt

Uploading part 2 of 2 to GCS

ℹ 2025-05-06 04:35:05.870814 > File size detected as  22.8 Mb

ℹ 2025-05-06 04:35:05.930243 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/pids-drg-vm/o/?uploadType=resumable&name=data%2Fphic%2Fthai%2Fpre%2Fchkpt_4_thai_grouper_input_2022_full_part_2_of_2.txt&upload_id=AAO2VwprWcR2Z_IBuhMaYU-nOpxPMRR1fm6IJ5sXqpT-ecNpV9k8FKl67bPD3WdM9v_XRrXbP9sWd7dcjGZDaYIDV5TP0jnpy_tcEvGtI4yvsu4



Reading final
Exporting for grouper
Classes ‘data.table’ and 'data.frame':	5000000 obs. of  42 variables:
 $ CASEID : chr  "1" "2" "3" "4" ...
 $ DOB    : chr  "20/01/1990" "03/02/1976" "21/01/1958" "19/01/2015" ...
 $ Sex    : num  1 1 1 2 2 2 2 2 1 1 ...
 $ DateAdm: chr  "20/01/2023" "03/02/2023" "21/01/2023" "19/01/2023" ...
 $ TimeAdm: chr  "2314" "1910" "1248" "2203" ...
 $ DateDsc: chr  "24/01/2023" "15/02/2023" "24/01/2023" "25/01/2023" ...
 $ TimeDsc: chr  "0911" "1434" "1139" "1748" ...
 $ DischT : Factor w/ 5 levels "1","2","3","4",..: 1 1 1 1 1 1 2 1 1 1 ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "K409" "L039" "--" "J189" ...
 $ SDx1   : chr  "--" "--" "--" "--" ...
 $ SDx2   : chr  "--" "--" "--" "--" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SDx9 

Saved part 1 of 2 to /home/resurreccion_cmc/pids-drg-claims/data-cleaning/data/chkpts/2025/chkpt_4_thai_master_input/chkpt_4_thai_grouper_input_2023_full_part_1_of_2.txt

Uploading part 1 of 2 to GCS

ℹ 2025-05-06 04:38:07.967979 > File size detected as  770.9 Mb

ℹ 2025-05-06 04:38:08.077524 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/pids-drg-vm/o/?uploadType=resumable&name=data%2Fphic%2Fthai%2Fpre%2Fchkpt_4_thai_grouper_input_2023_full_part_1_of_2.txt&upload_id=AAO2VwrzCHfXpPrPpC0tVacPcClbcFsR1DgsHWVEKbfulhobiMTb4S8wn-1keiM5uG5C2AMrflMkDsi-pZZV9uVL2k8fDj2qYstyXKhTa1AxII8



Classes ‘data.table’ and 'data.frame':	1230833 obs. of  42 variables:
 $ CASEID : chr  "5000001" "5000002" "5000003" "5000004" ...
 $ DOB    : chr  "30/03/2023" "31/03/2023" "22/04/1957" "30/04/2023" ...
 $ Sex    : num  2 2 1 1 1 1 2 1 2 2 ...
 $ DateAdm: chr  "30/03/2023" "31/03/2023" "22/04/2023" "30/04/2023" ...
 $ TimeAdm: chr  "0512" "1601" "0014" "1825" ...
 $ DateDsc: chr  "31/03/2023" "05/04/2023" "24/04/2023" "02/05/2023" ...
 $ TimeDsc: chr  "1821" "1158" "1758" "1208" ...
 $ DischT : Factor w/ 5 levels "1","2","3","4",..: 1 1 1 1 1 1 1 1 1 2 ...
 $ AdmWt  : chr  "2.795" "3.115" "--" "3.961" ...
 $ PDx    : chr  "--" "J189" "I639" "Z380" ...
 $ SDx1   : chr  "Z001" "A099" "--" "--" ...
 $ SDx2   : chr  "--" "E86" "--" "--" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SDx

Saved part 2 of 2 to /home/resurreccion_cmc/pids-drg-claims/data-cleaning/data/chkpts/2025/chkpt_4_thai_master_input/chkpt_4_thai_grouper_input_2023_full_part_2_of_2.txt

Uploading part 2 of 2 to GCS

ℹ 2025-05-06 04:38:45.076791 > File size detected as  190 Mb

ℹ 2025-05-06 04:38:45.137344 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/pids-drg-vm/o/?uploadType=resumable&name=data%2Fphic%2Fthai%2Fpre%2Fchkpt_4_thai_grouper_input_2023_full_part_2_of_2.txt&upload_id=AAO2Vwqz0JBP-Wq-hxqHSHALQw6KInlXNbmhOvsk7kA6lDKDDaRgcpmQiPzz5aZDMsRnDM6IZ9WCAqCk3mJ4ZGMz_dvE8bL1GLv0r_lY2w_9Mw



## Prompt


In [ ]:
# Prompt for manual confirmation if needed

response <- tolower(readline(
  prompt = "Have you run the Thai grouper manually? (y/n): "
))
if (response != "y") {
  stop("Thai Grouper not run yet. Continue on manually if necessary")
}
message("Continuing with the script...\n")


## Post-Processing


In [ ]:
if (!to_thai_all_years) {
  cat("\rDownloading Grouper results\n")
  flush.console()
  # Define chunk size
  cat("\rReading final\n")
  flush.console()
  result <- readRDS(here(
    chkpt_2_path,
    paste0(chkpt_2_prefix, year_to_load, suffix, "v2_part_c_ageday_bwt", ".rds")
  ))

  # str(result)
  result[, caseid := as.character(seq_len(nrow(result)))]
  result_mapping <- result[, .(id_series, caseid)]
  # Define chunk size
  chunk_size <- 5000000
  num_chunks <- ceiling(nrow(result) / chunk_size)

  # Download each part and combine them into thai_result
  thai_result <- list()
  for (i in seq_len(num_chunks)) {
    # Define the remote file name and local path for this part
    remote_file <- paste0(
      gcs_post_fpath,
      "/",
      paste0(
        chkpt_5_prefix, year_to_load, suffix,
        "part_", i, "_of_", num_chunks,
        "Res.TXT"
      )
    )
    local_file <- here(
      chkpt_5_path,
      paste0(
        chkpt_5_prefix, year_to_load, suffix,
        "part_", i, "_of_", num_chunks,
        "Res.TXT"
      )
    )

    # Download the part from GCS
    message("Downloading part ", i, " of ", num_chunks, " from GCS")
    gcs_get_object(
      object_name = remote_file,
      bucket = gcs_bucket,
      saveToDisk = local_file,
      overwrite = TRUE
    )

    # Read the downloaded part and store it in the list
    part_data <- fread(local_file, colClasses = "character")
    thai_result[[i]] <- part_data

    # Clean up memory
    rm(part_data)
    gc()
  }

  # Combine all parts into a single data.table
  thai_result <- rbindlist(thai_result, use.names = FALSE, fill = FALSE)
  cat(paste("nrow thai_result:", nrow(thai_result), "\n"))
  cat(paste("nrow result_mapping:", nrow(result_mapping), "\n"))
  cat(paste("nrow result:", nrow(result), "\n"))
  # Final message
  message("All parts downloaded and combined successfully.\n")
  if (to_debug) print(head(thai_result))
  # Check for duplicates in id_series
  if (any(duplicated(thai_result$caseid))) {
    # Identify duplicates
    duplicate_ids <- thai_result$caseid[duplicated(thai_result$caseid)]

    # Extract rows with duplicate id_series
    duplicate_rows <- thai_result[caseid %in% duplicate_ids, ]

    # Print rows with duplicates
    cat("Rows with duplicate 'id_series':\n")
    print(duplicate_rows)

    # Stop execution
    stop("The 'id_series' column contains duplicates. Execution stopped.")
  }

  thai_result <- merge(
    thai_result,
    result_mapping, # Select only caseid and id_series from result_mapping
    by = "caseid", # Column to join on
    all.x = TRUE,
    all.y = FALSE,
  )
  cat(paste("nrow thai_result:", nrow(thai_result), "\n"))
  cat(paste("nrow result_mapping:", nrow(result_mapping), "\n"))
  cat(paste("nrow result:", nrow(result), "\n"))
  cat("\rRenaming columns\n")
  flush.console()
  thai_result[, row := caseid]
  thai_result[, caseid := id_series]
  thai_result[, id_series := NULL]
  thai_result[, thai_drg := drg]
  thai_result[, thai_rw := rw]
  thai_result[, thai_wtlos := wtlos]
  thai_result[, thai_ot := ot]
  thai_result[, thai_adjrw := adjrw]
  thai_result[, thai_err := err]
  thai_result[, thai_warn := warn]
  thai_result[, thai_los := los]
  thai_result[, drg := NULL]
  thai_result[, drgname := NULL]
  thai_result[, rw := NULL]
  thai_result[, wtlos := NULL]
  thai_result[, ot := NULL]
  thai_result[, adjrw := NULL]
  thai_result[, err := NULL]
  thai_result[, warn := NULL]
  thai_result[, los := NULL]

  str(thai_result)
  print(nrow(thai_result))
  print(nrow(thai_result[thai_err == "6"]))

  # Please run thai grouper first
  result_after_thai <- data.table::copy(thai_result)
  result_after_thai[, id_series := caseid]
  result_after_thai[, caseid := NULL]
  result_after_thai[, thai_drg := as.character(thai_drg)]
  result_after_thai[, thai_rw := as.numeric(thai_rw)]
  result_after_thai[, thai_wtlos := as.numeric(thai_wtlos)]
  result_after_thai[, thai_ot := as.integer(thai_ot)]
  result_after_thai[, thai_adjrw := as.numeric(thai_adjrw)]
  result_after_thai[, thai_err := as.integer(thai_err)]
  result_after_thai[, thai_warn := as.integer(thai_warn)]
  result_after_thai[, thai_los := as.integer(thai_los)]
  # Reorder the columns in the result data.table to match the schema
  setcolorder(result_after_thai, c(
    "row",
    "id_series",
    "thai_drg",
    "thai_rw",
    "thai_wtlos",
    "thai_ot",
    "thai_adjrw",
    "thai_err",
    "thai_warn",
    "thai_los"
  ))
  print(result_after_thai[grepl("e", id_series)])
  # Check for duplicates in id_series
  if (any(duplicated(result_after_thai$id_series))) {
    # Identify duplicates
    duplicate_ids <- result_after_thai$id_series[
      duplicated(result_after_thai$id_series)
    ]

    # Extract rows with duplicate id_series
    duplicate_rows <- result_after_thai[id_series %in% duplicate_ids, ]

    # Print rows with duplicates
    cat("Rows with duplicate 'id_series':\n")
    print(duplicate_rows)

    # Stop execution
    stop("The 'id_series' column contains duplicates. Execution stopped.")
  }
  if (to_thai) print(nrow(result_after_thai))
  if (to_thai) print(result_after_thai[is.na(thai_drg)])
  if (to_thai) print(result_after_thai[is.na(id_series)])
  result_after_thai[, row := NULL]
  saveRDS(result_after_thai, here(
    chkpt_6_path,
    paste0(chkpt_6_prefix, year_to_load, suffix, ".rds")
  ))
} else if (to_thai_all_years) {
  # for (year_to_load in year_range) {
  for (year_to_load in year_range) {
    cat("\rDownloading Grouper results\n")
    flush.console()
    # Define chunk size
    cat("\rReading final\n")
    flush.console()
    result <- readRDS(here(
      chkpt_2_path,
      paste0(chkpt_2_prefix, year_to_load, suffix, "v2_part_c_ageday_bwt", ".rds")
    ))

    # str(result)
    result[, caseid := as.character(seq_len(nrow(result)))]
    result_mapping <- result[, .(id_series, caseid)]
    # Define chunk size
    chunk_size <- 5000000
    num_chunks <- ceiling(nrow(result) / chunk_size)

    # Download each part and combine them into thai_result
    thai_result <- list()
    for (i in seq_len(num_chunks)) {
      # Define the remote file name and local path for this part
      remote_file <- paste0(
        gcs_post_fpath,
        "/",
        paste0(
          chkpt_5_prefix, year_to_load, suffix,
          "part_", i, "_of_", num_chunks,
          "Res.TXT"
        )
      )
      local_file <- here(
        chkpt_5_path,
        paste0(
          chkpt_5_prefix, year_to_load, suffix,
          "part_", i, "_of_", num_chunks,
          "Res.TXT"
        )
      )

      # Download the part from GCS
      message("Downloading part ", i, " of ", num_chunks, " from GCS")
      gcs_get_object(
        object_name = remote_file,
        bucket = gcs_bucket,
        saveToDisk = local_file,
        overwrite = TRUE
      )

      # Read the downloaded part and store it in the list
      part_data <- fread(local_file, colClasses = "character")
      thai_result[[i]] <- part_data

      # Clean up memory
      rm(part_data)
      gc()
    }

    # Combine all parts into a single data.table
    thai_result <- rbindlist(thai_result, use.names = FALSE, fill = FALSE)
    cat(paste("nrow thai_result:", nrow(thai_result), "\n"))
    cat(paste("nrow result_mapping:", nrow(result_mapping), "\n"))
    cat(paste("nrow result:", nrow(result), "\n"))
    # Final message
    message("All parts downloaded and combined successfully.\n")
    if (to_debug) print(head(thai_result))
    # Check for duplicates in id_series
    if (any(duplicated(thai_result$caseid))) {
      # Identify duplicates
      duplicate_ids <- thai_result$caseid[duplicated(thai_result$caseid)]

      # Extract rows with duplicate id_series
      duplicate_rows <- thai_result[caseid %in% duplicate_ids, ]

      # Print rows with duplicates
      cat("Rows with duplicate 'id_series':\n")
      print(duplicate_rows)

      # Stop execution
      stop("The 'id_series' column contains duplicates. Execution stopped.")
    }

    thai_result <- merge(
      thai_result,
      result_mapping, # Select only caseid and id_series from result_mapping
      by = "caseid", # Column to join on
      all.x = TRUE,
      all.y = FALSE,
    )
    cat(paste("nrow thai_result:", nrow(thai_result), "\n"))
    cat(paste("nrow result_mapping:", nrow(result_mapping), "\n"))
    cat(paste("nrow result:", nrow(result), "\n"))
    cat("\rRenaming columns\n")
    flush.console()
    thai_result[, row := caseid]
    thai_result[, caseid := id_series]
    thai_result[, id_series := NULL]
    thai_result[, thai_drg := drg]
    thai_result[, thai_rw := rw]
    thai_result[, thai_wtlos := wtlos]
    thai_result[, thai_ot := ot]
    thai_result[, thai_adjrw := adjrw]
    thai_result[, thai_err := err]
    thai_result[, thai_warn := warn]
    thai_result[, thai_los := los]
    thai_result[, drg := NULL]
    thai_result[, drgname := NULL]
    thai_result[, rw := NULL]
    thai_result[, wtlos := NULL]
    thai_result[, ot := NULL]
    thai_result[, adjrw := NULL]
    thai_result[, err := NULL]
    thai_result[, warn := NULL]
    thai_result[, los := NULL]

    str(thai_result)
    print(nrow(thai_result))
    print(nrow(thai_result[thai_err == "6"]))

    # Please run thai grouper first
    result_after_thai <- data.table::copy(thai_result)
    result_after_thai[, id_series := caseid]
    result_after_thai[, caseid := NULL]
    result_after_thai[, thai_drg := as.character(thai_drg)]
    result_after_thai[, thai_rw := as.numeric(thai_rw)]
    result_after_thai[, thai_wtlos := as.numeric(thai_wtlos)]
    result_after_thai[, thai_ot := as.integer(thai_ot)]
    result_after_thai[, thai_adjrw := as.numeric(thai_adjrw)]
    result_after_thai[, thai_err := as.integer(thai_err)]
    result_after_thai[, thai_warn := as.integer(thai_warn)]
    result_after_thai[, thai_los := as.integer(thai_los)]
    # Reorder the columns in the result data.table to match the schema
    setcolorder(result_after_thai, c(
      "row",
      "id_series",
      "thai_drg",
      "thai_rw",
      "thai_wtlos",
      "thai_ot",
      "thai_adjrw",
      "thai_err",
      "thai_warn",
      "thai_los"
    ))
    print(result_after_thai[grepl("e", id_series)])
    # Check for duplicates in id_series
    if (any(duplicated(result_after_thai$id_series))) {
      # Identify duplicates
      duplicate_ids <- result_after_thai$id_series[
        duplicated(result_after_thai$id_series)
      ]

      # Extract rows with duplicate id_series
      duplicate_rows <- result_after_thai[id_series %in% duplicate_ids, ]

      # Print rows with duplicates
      cat("Rows with duplicate 'id_series':\n")
      print(duplicate_rows)

      # Stop execution
      stop("The 'id_series' column contains duplicates. Execution stopped.")
    }
    if (to_thai) print(nrow(result_after_thai))
    if (to_thai) print(result_after_thai[is.na(thai_drg)])
    if (to_thai) print(result_after_thai[is.na(id_series)])
    result_after_thai[, row := NULL]
    saveRDS(result_after_thai, here(
      chkpt_6_path,
      paste0(chkpt_6_prefix, year_to_load, suffix, ".rds")
    ))
  }
}
